# MEDA Preprocessing and Temporal Validation

In [1]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
OUTPUTS, MODELS = ROOT/'outputs', ROOT/'models'
sns.set_theme(style='whitegrid')

from src.meda_pipeline import build_features, validate_raw_schema
raw=pd.read_parquet(ROOT/'data/processed/stratified_training_sample.parquet'); X=build_features(raw); y=raw.PRESSURE.astype('float32')
print(X.shape, X.memory_usage(deep=True).sum()/2**20, 'MB')

(248092, 58) 54.89108657836914 MB


## Cleanup and feature contract

LMST/LTST are parsed into decimal hours and encoded with three Fourier harmonics. Solar longitude and rover angles receive sine/cosine encodings; mission sol receives Mars-year harmonics. Numeric fields are converted to float32, target and identifier columns are excluded, and row-level missing counts are retained. Models that cannot natively handle missing values use median imputation fitted only on training data.

In [2]:
indices=np.load(ROOT/'data/processed/temporal_indices_v2.npz'); splits={k:indices[k] for k in indices.files}
for name,idx in splits.items(): print(name,len(idx),raw.iloc[idx].sol.min(),raw.iloc[idx].sol.max())
assert set(splits['train']).isdisjoint(splits['validation']); assert set(splits['train']).isdisjoint(splits['test']); assert set(splits['validation']).isdisjoint(splits['test'])

train 170092 1.0 73.0
validation 39000 74.0 86.0
test 39000 87.0 100.0


## Validation rationale

The earliest partition is training (1-73), followed by validation (74-86) and an untouched final test (87-100). Rolling-origin folds operate only before the test boundary. Random validation was rejected because neighbouring observations share mission time and atmospheric state; it produced optimistic scores that did not transfer to the Kaggle period.

## Leakage and sampling limitations

The target and row identifier are removed before transformation. Preprocessing lives in `src/meda_pipeline.py` and is used unchanged by training, submission generation and Streamlit. The stratified sample caps each sol for an 8 GB laptop, so it improves temporal representation but is not identical to full-data training. The untouched test is still within the supplied training-era range and therefore cannot fully estimate the more distant Kaggle regime.